In [1]:
!pip install -q langchain langchain-community langchain-groq faiss-cpu tqdm

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 26.0 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 78.7 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 8.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 72.3 MB/s eta 0:00:00:00:01
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
xmanager 0.7.1 requires sqlalchemy==1.2.19, but you have sqlalchemy 2.0.45 which is incompatible.


In [2]:
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.embeddings import HuggingFaceEmbeddings
from langchain_groq import ChatGroq


# Reading files

In [3]:
import os
from tqdm import tqdm
from langchain.schema import Document
input_folder = "/kaggle/input/podcast/input"
transcripts = os.listdir(input_folder)
docs = []
for transcript in tqdm(transcripts, desc="reading ..."):
    with open(os.path.join(input_folder, transcript), "r") as f:
        transcript_content = f.read()

    docs.append(Document(
        page_content=transcript_content,
        metadata={"source":transcript}
    ))


reading ...: 100%|██████████| 1669/1669 [00:09<00:00, 173.27it/s]


# Chunking

In [4]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50,
    length_function=len,
    separators=["\n\n", "\n", " "]
)

chunks = text_splitter.split_documents(docs)

# Define Embedding model

In [5]:
model = HuggingFaceEmbeddings(
    model_name='sentence-transformers/all-MiniLM-L6-v2',
    model_kwargs={"device": "cuda"}
)

/tmp/ipykernel_55/450519994.py:1: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-huggingface package and should be used instead. To use it run `pip install -U :class:`~langchain-huggingface` and import as `from :class:`~langchain_huggingface import HuggingFaceEmbeddings``.
  model = HuggingFaceEmbeddings(
2025-12-25 12:55:30.955006: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1766667331.400489      55 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1766667331.517386      55 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already bee

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

# Define Vectorstore

In [6]:
from langchain_community.vectorstores import FAISS
vectorstores = FAISS.from_documents(
    documents=chunks,
    embedding=model
)

results = vectorstores.similarity_search(
    "What is AI?",
    k=2
)

In [7]:
response = "\n\n".join([res.page_content for res in results])
print(response)

Behind the Tech_Episode 35_StevenBathiche_Transcript.txt ’s hard to imagine AI in those terms, because to me it’s always felt like a tool and it’s, like, a tool to do this thing that you’ve been talking about in our conversation today. It’s like managing complexity. You use it to solve problems that are just too hard to solve any other way.

KEVIN SCOTT:  So, AI, on the one hand, is a[n] incredibly complicated assembly of technologies.  It's not just one thing, but maybe the simplest way to understand what it does, or like how to think about it is that AI is a tool that we built for automating tasks and doing work that would otherwise require a human being to do.


In [8]:
from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()
api_key = user_secrets.get_secret("GROQ_APIKEY")
 

llm = ChatGroq(
    model = "llama-3.3-70b-versatile",
    api_key=api_key
)

messages = [
    (
        "system",
        """You are my helpful QA assistant. Your role is to answer questions only based on the given content. Answer 'I do not know' if the provided content does not support a question, but do not generate from yourself. Answer politely and friendly.

        Content: {content}
        """.format(content=response)
    ),
    (
        "user",
        "What is AI?"
    )
]

llm_response = llm.invoke(messages)
print(llm_response)

content="According to the content, AI can be thought of as a tool that we built for automating tasks and doing work that would otherwise require a human being to do. It's also described as a tool to manage complexity and solve problems that are too hard to solve any other way." additional_kwargs={} response_metadata={'token_usage': {'completion_tokens': 57, 'prompt_tokens': 250, 'total_tokens': 307, 'completion_time': 0.153988187, 'completion_tokens_details': None, 'prompt_time': 0.01234573, 'prompt_tokens_details': None, 'queue_time': 0.008004888, 'total_time': 0.166333917}, 'model_name': 'llama-3.3-70b-versatile', 'system_fingerprint': 'fp_43d97c5965', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None} id='run--db56b694-301b-436b-bcd1-22488a046dd5-0' usage_metadata={'input_tokens': 250, 'output_tokens': 57, 'total_tokens': 307}


In [9]:
def get_answer(question: str) -> str:
    results = vectorstores.similarity_search(
    question,
    k=2
    )
    response = "\n\n".join([res.page_content for res in results])
    
    llm = ChatGroq(
    model = "llama-3.3-70b-versatile",
    api_key=api_key
    )

    messages = [
    (
        "system",
        """You are my helpful QA assistant. Your role is to answer questions only based on the given content. Answer 'I do not know' if the provided content does not support a question, but do not generate from yourself. Answer politely and friendly.

        Content: {content}
        """.format(content=response)
    ),
    (
        "user",
        "What is AI?"
    )
    ]

    llm_response = llm.invoke(messages)
    answer = llm_response.content
    return answer

In [10]:
print(get_answer("What is AI?"))

According to Kevin Scott, AI is a tool that we built for automating tasks and doing work that would otherwise require a human being to do. It's also described as an incredibly complicated assembly of technologies.


In [11]:
!pip install -q fastapi uvicorn faiss-cpu chainlit pyngrok

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.8/67.8 kB 2.3 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.8/9.8 MB 68.8 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.6/79.6 kB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 250.1/250.1 kB 15.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 456.8/456.8 kB 21.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.7/59.7 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 65.3/65.3 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.0/43.0 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.4/66.4 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 220.0/220.0 kB 13.1 MB/s eta 0:00:00

In [12]:
from fastapi import FastAPI
from pydantic import BaseModel 
app = FastAPI()

class Query(BaseModel):
    question: str

@app.post("/ask")
def ask(query: Query):
    try:
        result=get_answer(query.question)
        answer= result.content
    except Exception as e: 
        return {"answer": f"Error: {str(e)}"}
    return {"answer": answer}

In [13]:
%%writefile app.py
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.embeddings import HuggingFaceEmbeddings
from langchain_groq import ChatGroq
import os
from tqdm import tqdm
from langchain.schema import Document
input_folder = "/kaggle/input/podcast/input"
transcripts = os.listdir(input_folder)
docs = []
for transcript in tqdm(transcripts, desc="reading ..."):
    with open(os.path.join(input_folder, transcript), "r") as f:
        transcript_content = f.read()

    docs.append(Document(
        page_content=transcript_content,
        metadata={"source":transcript}
    ))
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50,
    length_function=len,
    separators=["\n\n", "\n", " "]
)

chunks = text_splitter.split_documents(docs)
model = HuggingFaceEmbeddings(
    model_name='sentence-transformers/all-MiniLM-L6-v2',
    model_kwargs={"device": "cuda"}
)
from langchain_community.vectorstores import FAISS
vectorstores = FAISS.from_documents(
    documents=chunks,
    embedding=model
)
from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()
api_key = user_secrets.get_secret("GROQ_APIKEY")

def get_answer(question: str) -> str:
    results = vectorstores.similarity_search(
    question,
    k=2
    )
    response = "\n\n".join([res.page_content for res in results])
    
    llm = ChatGroq(
    model = "llama-3.3-70b-versatile",
    api_key=api_key
    )

    messages = [
    (
        "system",
        """You are my helpful QA assistant. Your role is to answer questions only based on the given content. Answer 'I do not know' if the provided content does not support a question, but do not generate from yourself. Answer politely and friendly.

        Content: {content}
        """.format(content=response)
    ),
    (
        "user",
        "What is AI?"
    )
    ]

    llm_response = llm.invoke(messages) 
    return llm_response
    
from fastapi import FastAPI
from pydantic import BaseModel 
app = FastAPI()

class Query(BaseModel):
    question: str

@app.post("/ask")
def ask(query: Query):
    try:
        result=get_answer(query.question)
        answer= result.content
    except Exception as e: 
        return {"answer": f"Error: {str(e)}"}
    return {"answer": answer}

Writing app.py


In [14]:
%%writefile chainlit_app.py
import chainlit as cl
import requests

FASTAPI_URL = "http://localhost:8000/ask"

@cl.on_message
async def main(message: cl.Message):
    response = requests.post(
        FASTAPI_URL,
        json={"question": message.content}
    )
    answer = response.json()["answer"]

    await cl.Message(content=answer).send()

Writing chainlit_app.py


In [15]:
import subprocess
import time
 
subprocess.Popen(
    ["uvicorn", "app:app", "--host", "0.0.0.0", "--port", "8000"]
)

time.sleep(10)
 
subprocess.Popen(
    ["chainlit", "run", "chainlit_app.py", "--host", "0.0.0.0", "--port", "7860"]
)

print("FastAPI on :8000 | Chainlit on :7860")

/kaggle/working/app.py:2: LangChainDeprecationWarning: Importing HuggingFaceEmbeddings from langchain.embeddings is deprecated. Please replace deprecated imports:

>> from langchain.embeddings import HuggingFaceEmbeddings

with new imports of:

>> from langchain_community.embeddings import HuggingFaceEmbeddings
You can use the langchain cli to **automatically** upgrade many imports. Please see documentation here <https://python.langchain.com/docs/versions/v0_2/>
  from langchain.embeddings import HuggingFaceEmbeddings
reading ...: 100%|██████████| 1669/1669 [00:02<00:00, 758.64it/s]
/kaggle/working/app.py:26: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-huggingface package and should be used instead. To use it run `pip install -U :class:`~langchain-huggingface` and import as `from :class:`~langchain_huggingface import HuggingFaceEmbeddings``.


FastAPI on :8000 | Chainlit on :7860


In [16]:
from pyngrok import ngrok

from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()
ngrok_apikey = user_secrets.get_secret("ngrok-apikey")

ngrok.set_auth_token(ngrok_apikey)
public_url = ngrok.connect(7860) 
print("Public URL:", public_url)

Public URL: NgrokTunnel: "https://unfuming-excitingly-ruby.ngrok-free.dev" -> "http://localhost:7860"
2025-12-25 12:56:56 - INFO - chainlit - Created default chainlit markdown file at /kaggle/working/chainlit.md
2025-12-25 12:56:56 - INFO - chainlit - Your app is available at http://0.0.0.0:7860


INFO:     Started server process [178]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:8000 (Press CTRL+C to quit)


2025-12-25 12:57:11 - WARNING - chainlit - Translated markdown file for en-US not found. Defaulting to chainlit.md.
2025-12-25 12:57:12 - INFO - chainlit - Missing custom logo. Falling back to default logo.
INFO:     127.0.0.1:47822 - "POST /ask HTTP/1.1" 200 OK
